# 01-1 Token 是什么、从哪里来

大语言模型（如 ChatGPT、Qwen、Llama）看起来能"读懂"人类的语言，但它真正处理的并不是汉字或英文字母，而是一种叫做 **Token** 的最小单位。

本节我们将搞清楚：**Token 到底是什么？它是怎么从文本中切分出来的？不同模型的 Token 有什么区别？**

> 📝 我们将跟踪一句话在模型中的完整旅程：**"你爱我，我爱你，蜜雪"** → 这句话进入模型后，到底发生了什么？模型为什么能接出下一个字**"冰"**？

⏸ 关于多模态 Token（视觉、音频等），我们将在 [多模态 Token 探秘](./multimodal_tokens.ipynb) 中深入探讨。

## 0. 先看全貌：一句话的完整旅程

下面这段代码，把 **"你爱我，我爱你，蜜雪"** 这句话送入 Qwen3.5 模型，完整走一遍从输入到输出的全流程。

不需要理解每一行代码，先看结果——感受一下这句话在模型中经历了哪些变换：

In [1]:
from modelscope import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

tokenizer_id = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
model = AutoModelForCausalLM.from_pretrained(tokenizer_id)
model.eval()

text = "你爱我，我爱你，蜜雪"
print(f"输入文本: '{text}'")
print()

token_ids = tokenizer.encode(text)
decoded_tokens = [tokenizer.decode([tid]) for tid in token_ids]
print(f"[Step 1] 分词: {decoded_tokens}")
print(f"         Token IDs: {token_ids}")

input_ids = torch.tensor([token_ids])
with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)
print(f"[Step 2] Embedding: 形状 {embeddings.shape} (1句话, {embeddings.shape[1]}个Token, {embeddings.shape[2]}维向量)")

with torch.no_grad():
    outputs = model.model(input_ids)
    hidden_states = outputs.last_hidden_state
print(f"[Step 3] Transformer: 形状 {hidden_states.shape} (经过{model.config.num_hidden_layers}层变换)。小 tips，注意矩阵的维度没变化！！！")

with torch.no_grad():
    logits = model.lm_head(hidden_states[:, -1, :])
    probs = F.softmax(logits, dim=-1)
    next_id = torch.argmax(probs).item()
    next_text = tokenizer.decode([next_id])
    confidence = probs[0, next_id].item()

top5_probs, top5_ids = torch.topk(probs[0], 5)
top5_tokens = [tokenizer.decode([tid.item()]) for tid in top5_ids]

print(f"[Step 4] 预测: 下一个Token是 '{next_text}' (置信度={confidence:.2%})")
print(f"         Top-5 候选: {list(zip(top5_tokens, [f'{p:.2%}' for p in top5_probs.tolist()]))}")
print()
print(f"完整句子: '{text + next_text}'")
print()
print("=" * 60)
print("旅程回顾:")
print(f"  文本 '{text}'")
print(f"  -> 分词为 {len(token_ids)} 个 Token: {decoded_tokens}")
print(f"  -> 每个Token变成 {embeddings.shape[2]} 维向量")
print(f"  -> 经过 {model.config.num_hidden_layers} 层 Transformer 变换")
print(f"  -> 从 {len(tokenizer)} 个候选中选出概率最高的: '{next_text}'")
print(f"  -> 拼接得到: '{text + next_text}'")

2026-05-17 20:36:31,527 - modelscope - INFO - Target directory already exists, skipping creation.


2026-05-17 20:36:32,835 - modelscope - INFO - Target directory already exists, skipping creation.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

输入文本: '你爱我，我爱你，蜜雪'

[Step 1] 分词: ['你', '爱我', '，', '我爱你', '，', '蜜', '雪']
         Token IDs: [95933, 121196, 3709, 115734, 3709, 98735, 97055]
[Step 2] Embedding: 形状 torch.Size([1, 7, 1024]) (1句话, 7个Token, 1024维向量)
[Step 3] Transformer: 形状 torch.Size([1, 7, 1024]) (经过24层变换)。小 tips，注意矩阵的维度没变化！！！
[Step 4] 预测: 下一个Token是 '冰' (置信度=73.05%)
         Top-5 候选: [('冰', '73.05%'), ('儿', '14.36%'), ('卡', '1.42%'), ('天', '0.92%'), ('龙', '0.63%')]

完整句子: '你爱我，我爱你，蜜雪冰'

旅程回顾:
  文本 '你爱我，我爱你，蜜雪'
  -> 分词为 7 个 Token: ['你', '爱我', '，', '我爱你', '，', '蜜', '雪']
  -> 每个Token变成 1024 维向量
  -> 经过 24 层 Transformer 变换
  -> 从 248077 个候选中选出概率最高的: '冰'
  -> 拼接得到: '你爱我，我爱你，蜜雪冰'


看到了吗？一句话经过 **分词 → 向量化 → 深层变换 → 概率预测**，模型就能"猜"出下一个字。

接下来，我们逐步深入每个环节，搞清楚每一步到底在做什么、为什么这么做。

---

## 1. 切分：文本如何变成 Token？

模型的第一步，是把连续的文本切分成一个个离散的 **Token**。

Token 不是字，也不是词，而是介于两者之间的东西。让我们看看不同文本被切分后的效果：

In [2]:
texts = [
    "Tokenization is awesome!",
    "深度学习非常有趣。",
    "你爱我，我爱你，蜜雪",
]

for text in texts:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)
    decoded = [tokenizer.decode([tid]) for tid in token_ids]
    print(f"原文: {text}")
    print(f"  分词结果（词表里的样子）: {tokens}")
    print(f"  分词结果在词表里的id: {token_ids}")
    print(f"  分词结果还原: {decoded}")
    print(f"  Token数: {len(token_ids)}")
    print()

print("观察:")
print('  - 英文 "Tokenization" 被切成了 "Token" + "ization"（子词切分）')
print('  - 中文 "深度学习" 作为一个整体保留（高频词组）')
print('  - Token 可能是一个词、一个字、甚至一个词根——取决于训练语料中的出现频率')

原文: Tokenization is awesome!
  分词结果（词表里的样子）: ['Token', 'ization', 'Ġis', 'Ġawesome', '!']
  分词结果在词表里的id: [3214, 1954, 369, 12099, 0]
  分词结果还原: ['Token', 'ization', ' is', ' awesome', '!']
  Token数: 5

原文: 深度学习非常有趣。
  分词结果（词表里的样子）: ['æ·±åº¦åŃ¦ä¹ł', 'éĿŀå¸¸', 'æľīè¶£', 'ãĢĤ']
  分词结果在词表里的id: [124089, 96762, 101858, 1710]
  分词结果还原: ['深度学习', '非常', '有趣', '。']
  Token数: 4

原文: 你爱我，我爱你，蜜雪
  分词结果（词表里的样子）: ['ä½ł', 'çĪ±æĪĳ', 'ï¼Į', 'æĪĳçĪ±ä½ł', 'ï¼Į', 'èľľ', 'éĽª']
  分词结果在词表里的id: [95933, 121196, 3709, 115734, 3709, 98735, 97055]
  分词结果还原: ['你', '爱我', '，', '我爱你', '，', '蜜', '雪']
  Token数: 7

观察:
  - 英文 "Tokenization" 被切成了 "Token" + "ization"（子词切分）
  - 中文 "深度学习" 作为一个整体保留（高频词组）
  - Token 可能是一个词、一个字、甚至一个词根——取决于训练语料中的出现频率


### 等等，`tokenize()` 返回的中文怎么是乱码？

如果你用 `tokenizer.tokenize()` 而不是 `tokenizer.decode()`，会看到中文 Token 长这样：`æ·±åº¦åŃ¦ä¹ł`（其实是"深度学习"）。

这不是 bug！因为 Qwen 使用的是 **BBPE (Byte-level BPE)**，词表中的 Token 是字节级表示：
1. 中文先被编码为 UTF-8 字节（每个汉字 3 字节）
2. 字节值通过 Byte-to-Unicode 映射表变成可见字符（如 `0xE6` → `æ`）
3. BPE 算法在这些字节级 Token 上进行合并

也就是说`tokenize()` 返回的是词表中的**内部键名**，而 `decode()` 走的是反向路径（ID → 字节 → UTF-8解码 → 可读文本），能完美还原。

In [3]:
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

b2u = bytes_to_unicode()
u2b = {v: k for k, v in b2u.items()}

zh_text = "深度学习"
print(f"原始中文: {zh_text}")
print(f"字符数: {len(zh_text)}")

print(f"\n=== 第一步: UTF-8 编码 ===")
utf8_bytes = zh_text.encode('utf-8')
print(f"UTF-8 字节: {list(utf8_bytes)}")
print(f"字节数: {len(utf8_bytes)} (每个汉字 3 个字节)")

print(f"\n=== 第二步: 逐字节展示 ===")
for i, char in enumerate(zh_text):
    char_bytes = char.encode('utf-8')
    byte_str = ' '.join(f'0x{b:02X}' for b in char_bytes)
    unicode_str = ''.join(b2u[b] for b in char_bytes)
    print(f"  '{char}' -> 字节 [{byte_str}] -> Byte-to-Unicode 映射: '{unicode_str}'")

print(f"\n=== 第三步: 整体映射结果 ===")
full_mapped = ''.join(b2u[b] for b in utf8_bytes)
print(f"字节序列: {list(utf8_bytes)}")
print(f"Byte-to-Unicode 映射后: '{full_mapped}'")
print(f"这就是 tokenizer.tokenize() 返回的“乱码”!")

print(f"\n=== 第四步: 反向解码（“乱码” -> 中文）===")
recovered_bytes = bytes([u2b[c] for c in full_mapped])
recovered_text = recovered_bytes.decode('utf-8')
print(f"“乱码”字符串: '{full_mapped}'")
print(f"反向映射回字节: {list(recovered_bytes)}")
print(f"UTF-8 解码: '{recovered_text}'")
print(f"解码结果与原文一致: {recovered_text == zh_text}")

print(f"\n=== 完整轮转示意 ===")
print(f"中文 '深度学习' -> UTF-8 字节 [0xE6,0xB7,0xB1,0xE5,0xBA,0xA6,0xE5,0xAD,0xA6] -> Byte-to-Unicode 'æ·±åº¦åŃ¦' -> 反向映射+解码 -> '深度学习'")

原始中文: 深度学习
字符数: 4

=== 第一步: UTF-8 编码 ===
UTF-8 字节: [230, 183, 177, 229, 186, 166, 229, 173, 166, 228, 185, 160]
字节数: 12 (每个汉字 3 个字节)

=== 第二步: 逐字节展示 ===
  '深' -> 字节 [0xE6 0xB7 0xB1] -> Byte-to-Unicode 映射: 'æ·±'
  '度' -> 字节 [0xE5 0xBA 0xA6] -> Byte-to-Unicode 映射: 'åº¦'
  '学' -> 字节 [0xE5 0xAD 0xA6] -> Byte-to-Unicode 映射: 'åŃ¦'
  '习' -> 字节 [0xE4 0xB9 0xA0] -> Byte-to-Unicode 映射: 'ä¹ł'

=== 第三步: 整体映射结果 ===
字节序列: [230, 183, 177, 229, 186, 166, 229, 173, 166, 228, 185, 160]
Byte-to-Unicode 映射后: 'æ·±åº¦åŃ¦ä¹ł'
这就是 tokenizer.tokenize() 返回的“乱码”!

=== 第四步: 反向解码（“乱码” -> 中文）===
“乱码”字符串: 'æ·±åº¦åŃ¦ä¹ł'
反向映射回字节: [230, 183, 177, 229, 186, 166, 229, 173, 166, 228, 185, 160]
UTF-8 解码: '深度学习'
解码结果与原文一致: True

=== 完整轮转示意 ===
中文 '深度学习' -> UTF-8 字节 [0xE6,0xB7,0xB1,0xE5,0xBA,0xA6,0xE5,0xAD,0xA6] -> Byte-to-Unicode 'æ·±åº¦åŃ¦' -> 反向映射+解码 -> '深度学习'


### 为什么不直接用字或词？

| 方案 | 优点 | 缺点 |
| --- | --- | --- |
| **字/字母级** | 词表极小（英文26个） | 序列太长，难以捕捉语义 |
| **词级** | 语义完整 | 词表无限膨胀，遇到新词就傻了（OOV问题） |
| **子词级 (BPE)** | ✅ 兼顾词表大小和语义 | 需要训练分词器 |

BPE 的核心思想：**找到语料库中最常连续出现的字符组合，把它们合并成新的 Token。**
- 英文：`ing`、`tion`、`ly` 等常见词缀成为独立 Token，生僻词被拆成词根
- 中文：高频词组（如"中国"、"深度学习"）成为独立 Token，生僻字可能被拆为字节

## 2. 词表：模型"认识"多少个 Token？

每个 Tokenizer 都有一个固定的**词表**，存储了所有它"认识"的 Token。词表的大小和内容，直接决定了模型的"世界观"。

让我们看看 Qwen3.5 的词表长什么样，以及不同模型的词表差异如何影响分词效果：

In [4]:
print(f"模型: {tokenizer_id}")
print(f"词表大小 (vocab_size): {len(tokenizer)}")
print(f"模型最大输入长度: {tokenizer.model_max_length}")
print(f"\n--- 特殊 Token ---")
special_tokens = tokenizer.special_tokens_map
for name, value in special_tokens.items():
    token_id = tokenizer.convert_tokens_to_ids(value)
    print(f"  {name:15s} -> {value!r:20s}  (ID: {token_id})")

print(f"\n--- 词表中的 Token 示例 ---")
examples = ["hello", "世界", "ing", "Ġthe", "Ġis", "</w>"]
for ex in examples:
    tid = tokenizer.convert_tokens_to_ids(ex)
    is_unk = (tid is None) or (tid == tokenizer.unk_token_id)
    if is_unk:
        tokens_for_ex = tokenizer.tokenize(ex)
        actual_ids = tokenizer.encode(ex)
        decoded = [tokenizer.decode([i]) for i in actual_ids]
        print(f"  {ex!r:20s} -> 不在词表中！但 tokenize 后可以正常编码: {tokens_for_ex} (长度为{len(actual_ids)}) -> 解码: {decoded}")
    else:
        print(f"  {ex!r:20s} -> ID={tid}")

print(f"\n--- 词表 ID 范围 ---")
print(f"  最小 ID: 0, 最大 ID: {len(tokenizer) - 1}")
print(f"\n--- 词表首尾 Token ---")
for i in [0, 1, 2, 3, 4]:
    print(f"  ID {i}: {tokenizer.decode([i])!r}")
print(f"  ...")
for i in range(len(tokenizer) - 3, len(tokenizer)):
    print(f"  ID {i}: {tokenizer.decode([i])!r}")


模型: Qwen/Qwen3.5-0.8B
词表大小 (vocab_size): 248077
模型最大输入长度: 262144

--- 特殊 Token ---
  eos_token       -> '<|im_end|>'          (ID: 248046)
  pad_token       -> '<|endoftext|>'       (ID: 248044)
  audio_bos_token -> '<|audio_start|>'     (ID: 248070)
  audio_eos_token -> '<|audio_end|>'       (ID: 248071)
  audio_token     -> '<|audio_pad|>'       (ID: 248076)
  image_token     -> '<|image_pad|>'       (ID: 248056)
  video_token     -> '<|video_pad|>'       (ID: 248057)
  vision_bos_token -> '<|vision_start|>'    (ID: 248053)
  vision_eos_token -> '<|vision_end|>'      (ID: 248054)

--- 词表中的 Token 示例 ---
  'hello'              -> ID=14556
  '世界'                 -> 不在词表中！但 tokenize 后可以正常编码: ['ä¸ĸçķĮ'] (长度为1) -> 解码: ['世界']
  'ing'                -> ID=286
  'Ġthe'               -> ID=279
  'Ġis'                -> ID=369
  '</w>'               -> 不在词表中！但 tokenize 后可以正常编码: ['</', 'w', '>'] (长度为3) -> 解码: ['</', 'w', '>']

--- 词表 ID 范围 ---
  最小 ID: 0, 最大 ID: 248076

--- 词表首尾 Token ---
  ID 0

### 为什么"世界"不在词表中？—— BBPE 词表的内部表示

你可能会惊讶：像"世界"这样高频的中文词，居然显示"不在词表中"！
但这**并不意味着模型不认识"世界"**。原因在于 BBPE 词表的存储方式：

- **词表中的 Token 是字节级内部表示**，不是人类可读的文本。例如"世界"在词表中存储为 `ä¸ĸçķĮ`（UTF-8 字节经 Byte-to-Unicode 映射后的结果），而不是直接存"世界"两个字。
- `convert_tokens_to_ids("世界")` 查找的是**原始字符串"世界"**在词表中的位置，自然找不到。
- 但 `tokenizer.encode("世界")` 会先进行 BPE 分词，将"世界"切分为词表中存在的 Token（可能是一个整体 Token，也可能被拆分），然后返回对应的 ID。

所以，正确的理解是：**"世界"作为一个可编码的文本单位，模型是完全认识的**；只是它的词表"键名"用的是字节级表示，而不是人类可读的中文。

这也是为什么我们之前看到 `tokenize()` 对中文返回"乱码"——那正是词表中的真实键名。
而 `decode()` 方法能完美还原，因为它走的是"Token ID → 字节级表示 → UTF-8 解码 → 可读文本"的反向路径。

### 同一句话，不同模型切出不同结果

中国的 Qwen 和美国的 Gemma，对同一段中文的分词结果差异巨大——这直接影响 Token 消耗和 API 费用：

In [5]:
from modelscope import AutoTokenizer as MS_AutoTokenizer
import logging
logging.getLogger("modelscope").setLevel(logging.ERROR)

gemma_tokenizer = MS_AutoTokenizer.from_pretrained("google/gemma-4-E2B-it")

zh_texts = [
    "我爱你，中国！五星红旗迎风飘扬",
    "人工智能、机器学习、深度学习三者之间的关系",
    "春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。",
]
en_text = "Supercalifragilisticexpialidocious"

models = [
    ("Qwen3.5 (中国)", tokenizer),
    ("Gemma-4 (美国)", gemma_tokenizer),
]

for lang_label, txt_list in [("中文", zh_texts), ("英文", [en_text])]:
    for txt in txt_list:
        print(f"\n{'='*60}")
        print(f"{lang_label}: {txt}")
        print(f"{'='*60}")
        for model_label, tok in models:
            ids = tok.encode(txt)
            decoded_tokens = [tok.decode([tid]) for tid in ids]
            print(f"\n  [{model_label}]")
            print(f"  词表大小: {len(tok):,}  |  Token 数量: {len(ids)}")
            print(f"  逐 Token 解码: {decoded_tokens}")

print(f"\n\n=== 小结 ===")
print("中文场景: Qwen 的词表包含大量中文词组（如'五星红旗'、'迎风'等），同样的中文只需要更少的 Token。")
print("  例如'我爱你，中国！五星红旗迎风飘扬'：Qwen 只需 7 个 Token，Gemma 却需要 13 个！")
print("英文场景: 两个模型差异不大，但对超长词（如 Supercalifragilisticexpialidocious）的分词边界不同。")
print("这就是为什么使用非中文优化的大模型 API 时，中文 prompt 的 Token 消耗和费用通常比英文更高。")



中文: 我爱你，中国！五星红旗迎风飘扬

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 7
  逐 Token 解码: ['我爱你', '，', '中国', '！', '五星红旗', '迎风', '飘扬']

  [Gemma-4 (美国)]
  词表大小: 262,144  |  Token 数量: 13
  逐 Token 解码: ['我', '爱你', '，', '中国', '！', '五', '星', '红', '旗', '迎', '风', '飘', '扬']

中文: 人工智能、机器学习、深度学习三者之间的关系

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 7
  逐 Token 解码: ['人工智能', '、', '机器学习', '、', '深度学习', '三者', '之间的关系']

  [Gemma-4 (美国)]
  词表大小: 262,144  |  Token 数量: 10
  逐 Token 解码: ['人工智能', '、', '机器学习', '、', '深度', '学习', '三', '者', '之间的', '关系']

中文: 春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 19
  逐 Token 解码: ['春', '眠', '不觉', '晓', '，', '处处', '闻', '啼', '鸟', '。', '夜', '来', '风雨', '声', '，', '花落', '知', '多少', '。']

  [Gemma-4 (美国)]
  词表大小: 262,144  |  Token 数量: 23
  逐 Token 解码: ['春', '眠', '不', '觉', '晓', '，', '处', '处', '闻', '啼', '鸟', '。', '夜', '来', '风', '雨', '声', '，', '花', '落', '知', '多少', '。']

英文: Supercalifragilisticexpialidocious

  [Qwen3.5 (中国)]
  词表大小: 248,077  |  Token 数量: 11
 

---

## 小结

本节我们搞清楚了 Token 的来源：

1. **Token 不是字，也不是词**，而是子词级别的切分单元，由 BPE 算法从训练语料中自动学习得到
2. **BBPE 词表**使用字节级表示，中文 Token 在词表中存储为 UTF-8 字节映射后的"乱码"，但 `decode()` 可以完美还原
3. **不同模型的词表差异巨大**：Qwen 对中文更高效（7 个 Token vs Gemma 的 13 个），直接影响 API 费用

下一步：Token 是离散的整数 ID，神经网络无法直接计算——它是如何变成可计算的向量的？👉 [01-2 Token 为什么能被计算？](./01-2-token_why_computable.ipynb)